In [1]:
import pandas as pd
import joblib
import json
import os
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,recall_score, f1_score, confusion_matrix, classification_report)

In [2]:
X_train = pd.read_csv("data/processed/X_train.csv")
y_train = pd.read_csv("data/processed/y_train.csv").squeeze("columns")
X_test = pd.read_csv("data/processed/X_test.csv")
y_test = pd.read_csv("data/processed/y_test.csv").squeeze("columns")

print(X_train.shape, X_test.shape)

(31524, 63) (7880, 63)


In [3]:
model2 = RandomForestClassifier(
    n_estimators=400 ,
    max_depth=8 ,
    min_samples_leaf=5 ,
    class_weight="balanced" ,
    random_state=42 ,
    n_jobs=-1)
    

model2.fit(X_train, y_train)
print("Trained")

Trained


In [4]:
y_pred2 = model2.predict(X_test)
y_proba2 = model2.predict_proba(X_test)[:, 1]

In [5]:
accuracy2 = accuracy_score(y_test, y_pred2)
roc_auc2 = roc_auc_score(y_test, y_proba2)
precision2 = precision_score(y_test, y_pred2)
recall2 = recall_score(y_test, y_pred2)
f1_2 = f1_score(y_test, y_pred2)

print("Accuracy:", accuracy2)
print("ROC-AUC:", roc_auc2)
print("Precision:", precision2)
print("Recall:", recall2)
print("F1:", f1_2)

Accuracy: 0.5741116751269035
ROC-AUC: 0.713236735259068
Precision: 0.4138707037643208
Recall: 0.8046937151949085
F1: 0.5466090245879492


In [6]:
print(classification_report(y_test, y_pred2, target_names=["no_deposit", "deposit"]))

              precision    recall  f1-score   support

  no_deposit       0.84      0.47      0.60      5366
     deposit       0.41      0.80      0.55      2514

    accuracy                           0.57      7880
   macro avg       0.62      0.64      0.57      7880
weighted avg       0.70      0.57      0.58      7880



In [7]:
os.makedirs("models" , exist_ok=True)
joblib.dump(model2, "models/model2_random_forest.joblib")
print("Model saved")

Model saved


In [8]:
predictions_df2 = X_test.copy()
predictions_df2["y_true"] = y_test.values
predictions_df2["y_pred"] = y_pred2
predictions_df2["y_proba"] = y_proba2
predictions_df2[["y_true", "y_pred", "y_proba"]].to_csv("data/processed/model2_predictions.csv" , index=False)

m2_metrics ={"accuracy":accuracy2,
           "roc_auc":roc_auc2,
           "precision":precision2,
           "recall":recall2,
           "f1": f1_2}
with open("data/processed/model2_metrics.json", "w") as f:
    json.dump(m2_metrics, f, indent=2)

print("saved predictions and metrics")


saved predictions and metrics


In [ ]:
y_t, y_p, y_pr = np.asarray(y_test), np.asarray(y_pred2), np.asarray(y_proba2)
print(confusion_matrix(y_t, y_p))

rng = np.random.default_rng(42)
auc, rec, f1s = [], [], []
for _ in range(1000):
    i = rng.integers(0, len(y_t), len(y_t))
    auc.append(roc_auc_score(y_t[i], y_pr[i]))
    rec.append(recall_score(y_t[i], y_p[i]))
    f1s.append(f1_score(y_t[i], y_p[i]))

for name, v in [("ROC-AUC", auc), ("Recall", rec), ("F1", f1s)]:
    lo, hi = np.percentile(v, [2.5, 97.5])
    print(f"{name}: 95% CI [{lo:.3f}, {hi:.3f}]")

[[2501 2865]
 [ 491 2023]]
